
# Real Estate Housing Price Prediction
## Ridge & Lasso Regression

### Project Overview
This project aims to predict median house values using the California Housing dataset.

The project involves exploratory data analysis, data preprocessing, feature engineering, and the implementation of multiple regression models.

### Objectives
- Explore and understand the California Housing dataset.
- Handle missing values and categorical variables.
- Perform feature engineering and investigate multicollinearity.
- Implement Ordinary Least Squares (OLS), Ridge, and Lasso Regression.
- Optimize model hyperparameters using GridSearchCV.
- Evaluate and compare models using RMSE, MAE, and R².
- Analyze prediction errors through residual visualizations.

### Technologies Used
Python, Pandas, NumPy, Matplotlib, Seaborn, and Scikit-learn.


In [1]:

# ==========================================
# 1. IMPORT REQUIRED LIBRARIES
# ==========================================

# Data manipulation and numerical operations
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Model evaluation metrics
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("All required libraries imported successfully!")


All required libraries imported successfully!



## 2. Loading the Dataset

The California Housing dataset is loaded into a Pandas DataFrame for further analysis.

The dataset contains information about housing characteristics, population, household statistics, geographical location, and median house values.


In [2]:

# ==========================================
# 2.1 LOAD THE DATASET
# ==========================================

from pathlib import Path

# Locate the dataset from common notebook working directories
possible_paths = [
    Path("../dataset/housing.csv"),
    Path("dataset/housing.csv"),
    Path("Task_2_Housing_Price_Prediction/dataset/housing.csv")
]

dataset_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if dataset_path is None:
    raise FileNotFoundError(
        "housing.csv not found. Please check the dataset folder."
    )

# Load the CSV file
df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print(f"Dataset location: {dataset_path}")


Dataset loaded successfully!
Dataset location: ..\dataset\housing.csv



### 2.2 Previewing the Dataset

The first five records are displayed to understand how the data is organized and to examine the available features.


In [3]:

# Display the first five rows
df.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.230,37.880,41.000,880.000,129.000,322.000,126.000,8.325,452600.000,NEAR BAY
1,-122.220,37.860,21.000,7099.000,1106.000,2401.000,1138.000,8.301,358500.000,NEAR BAY
2,-122.240,37.850,52.000,1467.000,190.000,496.000,177.000,7.257,352100.000,NEAR BAY
3,-122.250,37.850,52.000,1274.000,235.000,558.000,219.000,5.643,341300.000,NEAR BAY
4,-122.250,37.850,52.000,1627.000,280.000,565.000,259.000,3.846,342200.000,NEAR BAY



### 2.3 Dataset Dimensions

The shape attribute is used to determine the number of observations and features present in the dataset.


In [4]:

# Display the number of rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Dataset dimensions:", df.shape)


Number of rows: 20640
Number of columns: 10
Dataset dimensions: (20640, 10)



### 2.4 Dataset Information

The column names and data types are examined to distinguish between numerical and categorical variables.


In [5]:

# Display column names
print("Column Names:")
print(df.columns.tolist())

# Display dataset information
print("\nDataset Information:")
df.info()


Column Names:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.7 MB



### 2.5 Descriptive Statistical Summary

Descriptive statistics are generated to understand the distribution of numerical features, including their mean, standard deviation, minimum, maximum, and quartile values.


In [6]:

# Generate descriptive statistics
df.describe()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000,20640.000,20640.000,20640.000,20433.000,20640.000,20640.000,20640.000,20640.000
mean,-119.570,35.632,28.639,2635.763,537.871,1425.477,499.540,3.871,206855.817
std,2.004,2.136,12.586,2181.615,421.385,1132.462,382.330,1.900,115395.616
min,-124.350,32.540,1.000,2.000,1.000,3.000,1.000,0.500,14999.000
25%,-121.800,33.930,18.000,1447.750,296.000,787.000,280.000,2.563,119600.000
50%,-118.490,34.260,29.000,2127.000,435.000,1166.000,409.000,3.535,179700.000
75%,-118.010,37.710,37.000,3148.000,647.000,1725.000,605.000,4.743,264725.000
max,-114.310,41.950,52.000,39320.000,6445.000,35682.000,6082.000,15.000,500001.000



### 2.6 Missing Value Analysis

The dataset is examined for missing values to identify columns that require data imputation during preprocessing.


In [7]:

# Count missing values in each column
missing_values = df.isnull().sum()

# Calculate missing value percentages
missing_percentage = (missing_values / len(df)) * 100

# Create a summary table
missing_summary = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage
})

missing_summary


,Missing Values,Missing Percentage
longitude,0,0.000
latitude,0,0.000
housing_median_age,0,0.000
total_rooms,0,0.000
total_bedrooms,207,1.003
population,0,0.000
households,0,0.000
median_income,0,0.000
median_house_value,0,0.000
ocean_proximity,0,0.000



### 2.7 Duplicate Record Analysis

The dataset is checked for duplicate observations to identify repeated records that may affect the analysis.


In [8]:

# Count duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)


Number of duplicate records: 0



### 2.8 Categorical Feature Analysis

The unique categories and their frequency distribution are examined for the ocean_proximity feature.


In [9]:

# Display unique categories
print("Unique ocean proximity categories:")
print(df['ocean_proximity'].unique())

# Display category frequencies
print("\nCategory Distribution:")
print(df['ocean_proximity'].value_counts())


Unique ocean proximity categories:
<ArrowStringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

Category Distribution:
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64



### 2.9 Initial Dataset Observations

The initial inspection of the California Housing dataset revealed the following:

1. The dataset contains 20,640 observations and 10 columns.
2. It consists of 9 numerical features and 1 categorical feature.
3. The target variable for prediction is `median_house_value`.
4. The `total_bedrooms` column contains 207 missing values.
5. The `ocean_proximity` feature contains multiple categorical values that require encoding.
6. The dataset requires further preprocessing before regression modeling.

These observations will guide the exploratory data analysis and preprocessing stages.
